# Fig. — Scalability in the locally low-rank regime (ADMM headline)

Interactive front-end. Loader/summary helpers come from `build_fig_lowrank.py`;
**`build_figure` is inlined below as an editable cell** so you can override
`ALGORITHM_STYLE`, the shaded region, or layout without editing the module.

- **(top)** min-SINR vs N_ue; **(bottom)** sum-SCNR vs N_ue.
- Per-AP DoF is M = antennas/AP. Once **N_ue > M** the local channels are locally
  low-rank: CORDIS-Split (fixed local BF) can't null all co-users from one AP and
  diverges, while CORDIS-ADMM holds QoS via cross-AP consensus and tracks Centralized.
- The N_ue > M region is shaded; N_ue = M is the full-rank reference.

**Compatible experiment:** `n_ue_sweep` (`kind=='sweep'`), run at the low-rank config
(`n_ap=11, n_ant=3, n_rf_chains=3, n_targets=1`).

In [ ]:
# Make the builder + cordis importable, then apply the paper rcParams.
import sys, logging
from pathlib import Path
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

FIG_DIR = Path.cwd()
if str(FIG_DIR) not in sys.path:
    sys.path.insert(0, str(FIG_DIR))

import build_fig_lowrank as B   # loader + helpers (single source of truth)
from cordis.plotting import apply_paper_style, figsize, plot_sweep, save_figure
from cordis.plotting.style import ALGORITHM_STYLE   # tweak here to restyle

USE_TEX = True   # set False on a node without pdflatex
apply_paper_style()
logging.basicConfig(level=logging.INFO, format='%(levelname)-7s %(message)s')

## 1. Load the low-rank N_ue-sweep run

`RESULT_DIR = None` auto-picks the newest run (`array_<jobid>_aggregated/` preferred;
`latest` symlink never used). This must be the **low-rank** campaign (n_ant=3), not the
favorable-config n_ue_sweep.

In [ ]:
RESULT_DIR = None   # e.g. 'results/exp_n_ue_sweep/array_12345_aggregated'
result, result_dir = B.load_sweep(RESULT_DIR, experiment='n_ue_sweep')
print('run:', result_dir)
print('N_ue grid:', [int(k) for k in sorted(result.sweep_results.keys())])

## 2. Resolve algorithms, SCNR metric, per-AP DoF M

`RANK_THRESHOLD = None` auto-detects M (= n_ant) from the run metadata; override if
the metadata doesn't carry it. The shaded low-rank region is N_ue > M.

In [ ]:
RANK_THRESHOLD = None                  # None -> detect M (=n_ant) from metadata
ONLY           = list(B.PREFERRED)     # Centralized / CORDIS-ADMM / CORDIS-Split
SCNR_METRIC    = None                  # None -> first available of B.SCNR_METRIC_PREFERENCE

rank_threshold, detected = (RANK_THRESHOLD, True) if RANK_THRESHOLD is not None \
    else B._detect_rank_threshold(result)
only = B._present_algorithms(result, ONLY)
scnr_metric = SCNR_METRIC or B._select_scnr_metric(result, only=only)
print(f'M (per-AP DoF) = {rank_threshold}  (detected={detected})')
print('algorithms:', only)
print('scnr metric:', scnr_metric)

## 3. Numeric summary (caption sanity check)

Compares each algorithm at the full-rank reference (N_ue≈M) and at max load, and
prints the headline min-SINR gap CORDIS-ADMM − CORDIS-Split at the most-loaded point.

In [ ]:
summary = B.collect_summary(result, only, scnr_metric, rank_threshold)
B._print_summary(result, summary, scnr_metric, rank_threshold, detected)

## 4. `build_figure` — editable copy

The **exact** function from `build_fig_lowrank.py`, inlined so you can edit it here
(restyle, change the shaded region, relabel) and re-run. Copy it back into the module
to make an edit permanent; or mutate `ALGORITHM_STYLE` in the setup cell to restyle
without editing the body.

In [ ]:
# Bind the module-level names the function body references, so the inlined
# copy behaves identically to B.build_figure.
from typing import Optional, Sequence   # the inlined signature uses these
PREFERRED               = B.PREFERRED
SINR_METRIC             = B.SINR_METRIC
RANK_THRESHOLD_DEFAULT  = B.RANK_THRESHOLD_DEFAULT
_SCNR_YLABEL            = B._SCNR_YLABEL
_present_algorithms     = B._present_algorithms
_select_scnr_metric     = B._select_scnr_metric
logger                  = logging.getLogger('fig_lowrank.nb')

In [ ]:
def build_figure(result,
                 *,
                 only: Optional[Sequence[str]] = None,
                 scnr_metric: Optional[str] = None,
                 rank_threshold: int = RANK_THRESHOLD_DEFAULT,
                 shade_lowrank: bool = True,
                 use_tex: bool = True):
    """Assemble the stacked low-rank scalability figure; return
    ``(fig, only, scnr_metric)``.

    Built directly on ``cordis.plotting.plot_sweep`` so the paper builder stays
    decoupled from scripts/.  Reuses the project per-algorithm style.
    """
    import matplotlib
    if use_tex is False:
        matplotlib.rcParams["text.usetex"] = False
    import matplotlib.pyplot as plt
    from cordis.plotting import apply_paper_style, figsize, plot_sweep

    apply_paper_style()
    if use_tex is False:
        matplotlib.rcParams["text.usetex"] = False

    only = list(only) if only else _present_algorithms(result, PREFERRED)
    if scnr_metric is None:
        scnr_metric = _select_scnr_metric(result, only=only)

    axis = result.sweep_axis
    xlabel = axis.display or r"$N_{\rm UE}$"
    keys = sorted(result.sweep_results.keys())
    x_lo, x_hi = keys[0], keys[-1]

    fig, (ax_top, ax_bot) = plt.subplots(
        2, 1, figsize=figsize(width="single", aspect=3.5 / 2.6),
        sharex=True, gridspec_kw={"hspace": 0.12},
    )

    # (top) min-SINR vs N_ue
    plot_sweep(result.sweep_results, metric=SINR_METRIC, ax=ax_top,
               xlabel="", ylabel=r"min-SINR [dB]", only=only, log_x=False)
    ax_top.set_title("Scalability in the locally low-rank regime")

    # (bot) SCNR vs N_ue
    if scnr_metric is not None:
        plot_sweep(result.sweep_results, metric=scnr_metric, ax=ax_bot,
                   xlabel=xlabel,
                   ylabel=_SCNR_YLABEL.get(scnr_metric, scnr_metric),
                   only=only, log_x=False)
    else:
        logger.warning("No SCNR metric available; bottom panel left empty.")
        ax_bot.set_xlabel(xlabel)

    # integer N_ue ticks + a little x-margin
    for ax in (ax_top, ax_bot):
        ax.set_xticks([int(k) for k in keys])
        ax.set_xlim(x_lo - 0.3, x_hi + 0.3)

    # shade the locally-low-rank region  N_ue > M  on both panels
    shade_left = rank_threshold + 0.5
    if shade_lowrank and shade_left < x_hi + 0.3:
        for ax in (ax_top, ax_bot):
            ax.axvspan(max(shade_left, x_lo - 0.3), x_hi + 0.3,
                       color="0.85", alpha=0.5, zorder=0, linewidth=0)
        ax_top.text(0.99, 0.04,
                    r"locally low-rank ($N_{\rm UE} > M$)",
                    transform=ax_top.transAxes, ha="right", va="bottom",
                    fontsize="x-small", color="0.35")

    # one legend only (top); drop the duplicate on the bottom panel
    if ax_bot.get_legend() is not None:
        ax_bot.get_legend().remove()

    # NB: no fig.tight_layout() — conflicts with the shared-x / hspace stacked
    # layout.  save_figure() uses bbox_inches='tight'.
    return fig, only, scnr_metric


## 5. Render

In [ ]:
fig, used_only, used_scnr_metric = build_figure(
    result, only=only, scnr_metric=scnr_metric,
    rank_threshold=rank_threshold, use_tex=USE_TEX,
)
plt.show()

## 6. Save to `paper/figures/`

In [ ]:
out_stem = FIG_DIR.parents[1] / 'figures' / 'fig_lowrank'
paths = save_figure(
    fig, out_stem, formats=('pdf',),
    metadata={
        'Figure': 'fig_lowrank',
        'RankThresholdM': str(rank_threshold),
        'Algorithms': ', '.join(used_only),
        'ScnrMetric': str(used_scnr_metric),
        'Run': result_dir.name,
    },
)
for p in paths:
    print('wrote', p)